# MOCK #2 — Click-to-WhatsApp Ads (CTWA) — L6 10/10 Pass

**This notebook is your interview playbook.** Every question has 4 layers:
1. **WHAT TO SAY** — the 30-60 sec answer the interviewer wants to hear
2. **WHY** — the L6 reasoning: constraints, scale, tradeoffs
3. **HOW** — the working code
4. **EDGE CASES / TESTS** — what could go wrong

Covers the 4 rubric parts at L6 (Staff) level:
- **Part A** — Data Modeling (5 tables, SCD2, partition, dedup, privacy, late data)
- **Part B** — SQL (Q1 7-day ROAS, Q2 5-min reply SLA, plus 6 more)
- **Part C** — Coding (Python ghosted threads + 10B scale extension)
- **Part D** — Architecture (end-to-end flow, DQ, cost, iOS 14.5)

Plus: cheat sheet, practice tip, and live drill cells.

In [10]:
# ============================================================
# SETUP
# ============================================================
import sys, subprocess
try:
    from pyspark.sql import SparkSession
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.1"])
    from pyspark.sql import SparkSession

spark = (SparkSession.builder.master("local[*]")
         .appName("ctwa-10of10")
         .config("spark.sql.shuffle.partitions","4")
         .config("spark.ui.showConsoleProgress","false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

from datetime import datetime
from pyspark.sql.types import *

# ---- Mock data matching the exact prompt tables ----
clicks = [
    ("c1","ad_10","adv_1",datetime(2026,5,1,10,0), "2026-05-01"),
    ("c2","ad_10","adv_1",datetime(2026,5,1,10,5), "2026-05-01"),
    ("c3","ad_11","adv_1",datetime(2026,5,1,11,0), "2026-05-01"),
    ("c4","ad_20","adv_2",datetime(2026,5,1,12,0), "2026-05-01"),
    ("c5","ad_10","adv_1",datetime(2026,4,20,10,0),"2026-04-20"),  # out of 7d window
    ("c99","ad_10","adv_1",datetime(2026,5,1,10,0),"2026-05-01"),  # duplicate
    ("c99","ad_10","adv_1",datetime(2026,5,1,10,0),"2026-05-01"),
]
threads = [
    ("t1","adv_1","h_u1","c1", datetime(2026,5,1,10,1), "2026-05-01"),
    ("t2","adv_1","h_u2","c2", datetime(2026,5,1,10,6), "2026-05-01"),
    ("t3","adv_1","h_u3",None,  datetime(2026,5,1,10,30),"2026-05-01"),  # organic (NULL)
    ("t4","adv_2","h_u4","c4", datetime(2026,5,1,12,1), "2026-05-01"),
    ("t5","adv_1","h_u5","c5", datetime(2026,4,20,10,2),"2026-04-20"),
]
messages = [
    ("m1","t1","user",    datetime(2026,5,1,10,1), "2026-05-01"),
    ("m2","t1","business",datetime(2026,5,1,10,20),"2026-05-01"),  # 19 min
    ("m3","t2","user",    datetime(2026,5,1,10,6), "2026-05-01"),
    ("m4","t2","business",datetime(2026,5,1,10,10),"2026-05-01"),  # 4 min
    ("m5","t3","user",    datetime(2026,5,1,10,30),"2026-05-01"),
    ("m6","t3","business",datetime(2026,5,1,10,32),"2026-05-01"),  # 2 min (organic)
    ("m7","t4","user",    datetime(2026,5,1,12,1), "2026-05-01"),
    ("m8","t4","user",    datetime(2026,5,2,12,30),"2026-05-02"),  # ghosted
]
conversions = [
    ("conv1","t1","ORDER_PLACED",datetime(2026,5,1,10,20),100.0,"2026-05-01"),
    ("conv2","t2","ORDER_PLACED",datetime(2026,5,3,9,0),   50.0,"2026-05-03"),  # late but <7d
    ("conv3","t3","ORDER_PLACED",datetime(2026,5,1,11,0),  30.0,"2026-05-01"),  # organic
    ("conv4","t5","ORDER_PLACED",datetime(2026,5,10,9,0), 999.0,"2026-05-10"),  # out of 7d
    ("conv5","t4","ORDER_PLACED",datetime(2026,5,1,18,0), 200.0,"2026-05-01"),  # ghosted but converted
]
ads = [
    ("ad_10","camp_1","adv_1","CBO",True, datetime(2026,4,15), datetime(9999,12,31)),
    ("ad_11","camp_1","adv_1","CBO",True, datetime(2026,4,15), datetime(9999,12,31)),
    ("ad_20","camp_2","adv_2","ABO",True, datetime(2026,4,15), datetime(9999,12,31)),
    ("ad_10","camp_1","adv_1","ABO",False,datetime(2026,1,1),  datetime(2026,4,15)),  # SCD2 history
]

fct_ad_clicks = spark.createDataFrame(clicks,
    "click_id STRING, ad_id STRING, advertiser_id STRING, click_time TIMESTAMP, ds STRING").cache()
fct_wa_threads = spark.createDataFrame(threads,
    "thread_id STRING, advertiser_id STRING, wa_user_hash STRING, attributed_click_id STRING, thread_start_time TIMESTAMP, ds STRING").cache()
fct_wa_messages_meta = spark.createDataFrame(messages,
    "message_id STRING, thread_id STRING, sender_type STRING, message_time TIMESTAMP, ds STRING").cache()
fct_merchant_conversions = spark.createDataFrame(conversions,
    "conversion_id STRING, thread_id STRING, conversion_type STRING, conversion_time TIMESTAMP, order_value_usd DOUBLE, ds STRING").cache()
dim_ad = spark.createDataFrame(ads,
    "ad_id STRING, campaign_id STRING, advertiser_id STRING, budget_type STRING, is_current BOOLEAN, valid_from TIMESTAMP, valid_to TIMESTAMP").cache()

for n, df in [("fct_ad_clicks",fct_ad_clicks),("fct_wa_threads",fct_wa_threads),
             ("fct_wa_messages_meta",fct_wa_messages_meta),
             ("fct_merchant_conversions",fct_merchant_conversions),
             ("dim_ad",dim_ad)]:
    df.createOrReplaceTempView(n)
print("OK 5 tables ready")

OK 5 tables ready


---
# PART A — DATA MODELING

## PRACTICE TIP (memorize): When the interviewer shows you a table on the board, **immediately redraw it as Input -> Output on your side**. It buys you 30 seconds of think-time AND shows structured thinking. The L6 interviewer is watching HOW you process ambiguity, not just the answer.

## WHAT TO SAY FIRST (the 30-sec opener — memorize word-for-word):

> "**Success is advertiser ROI**: conversion rate, time-to-first-reply, order value.
> **Constraint**: WhatsApp messages are **E2E encrypted** — I can only use **metadata**, not content.
> **Identity stitching** must use **`click_id`**, not `user_id`, because we cannot legally or technically join FB PII to WA PII."

## 5 TABLES — what to say for each:

### 1. `dim_ad` — SCD Type 2
> "SCD2 with `is_current`, `valid_from`, `valid_to` so we keep full history of `budget_type` changes (CBO↔ABO). SCD1 loses history; SCD3 only keeps prev/curr. SCD2 is correct for time-travel queries on ad metadata."

### 2. `fct_ad_clicks` — 1 row per CTWA ad click
> "Grain = 1 click. PK = `click_id` (the only allowed bridge via deep link). `advertiser_id` is **denormalized** to avoid joins on every ROAS query at 10B rows. `fb_user_hash` is hashed, never raw. Partition by `ds`, cluster by `advertiser_id` for per-advertiser scan locality."

### 3. `fct_wa_threads` — 1 row per WA thread started
> "The **bridge table**. `attributed_click_id` is nullable — NULL means organic thread (user opened WA without clicking an ad). We **cannot** join `wa_user_hash` to `fb_user_hash` even though it would be nice — that would break the E2E promise."

### 4. `fct_wa_messages_meta` — 1 row per message **METADATA ONLY**
> "**No `content` column** — and that's the L6 signal. E2E means our schema **must not carry** message bodies, even if we never populate it. Having the column would be a policy violation. Sender type {user, business} is the only signal we have about who's talking."

### 5. `fct_merchant_conversions` — 1 row per merchant-marked conversion
> "Grain = 1 conversion. `conversion_type` ∈ {ORDER_PLACED, ORDER_PAID, CANCELED}. Merchant can mark late up to **28 days**. This table is **mutable** — merchant may unmark — so we treat it as upsert keyed on `conversion_id`."

## THE 4 L6 POINTS YOU MUST SAY:
1. **Partition by `ds`, cluster by `advertiser_id`** on every fact table.
2. **Dedup** with `ROW_NUMBER() OVER (PARTITION BY click_id ORDER BY processing_time DESC) = 1` for idempotency on all 3 unique keys.
3. **Privacy**: no FB→WA PII join, only `click_id`. `is_deleted` flag for WA user erasure (GDPR).
4. **Late data**: 28-day lookback re-aggregation every night. Older partitions are immutable.

In [11]:
# ---- TEST: SCD2 point-in-time (the L6 differentiator) ----
print("=== dim_ad SCD2 (current + historical) ===")
spark.sql("SELECT ad_id, budget_type, is_current, valid_from, valid_to FROM dim_ad ORDER BY ad_id, valid_from").show(truncate=False)

rows_april = spark.sql("""
    SELECT ad_id, budget_type FROM dim_ad
    WHERE ad_id = 'ad_10' AND TIMESTAMP '2026-04-01' BETWEEN valid_from AND valid_to
""").collect()
rows_may = spark.sql("""
    SELECT ad_id, budget_type FROM dim_ad
    WHERE ad_id = 'ad_10' AND TIMESTAMP '2026-05-01' BETWEEN valid_from AND valid_to
""").collect()
assert rows_april[0]['budget_type'] == 'ABO', f"April should be ABO, got {rows_april}"
assert rows_may[0]['budget_type'] == 'CBO', f"May should be CBO, got {rows_may}"
print("OK ad_10 was ABO in April, CBO in May (SCD2 point-in-time works)")

=== dim_ad SCD2 (current + historical) ===
+-----+-----------+----------+-------------------+-------------------+
|ad_id|budget_type|is_current|valid_from         |valid_to           |
+-----+-----------+----------+-------------------+-------------------+
|ad_10|ABO        |false     |2026-01-01 00:00:00|2026-04-15 00:00:00|
|ad_10|CBO        |true      |2026-04-15 00:00:00|9999-12-31 00:00:00|
|ad_11|CBO        |true      |2026-04-15 00:00:00|9999-12-31 00:00:00|
|ad_20|ABO        |true      |2026-04-15 00:00:00|9999-12-31 00:00:00|
+-----+-----------+----------+-------------------+-------------------+

OK ad_10 was ABO in April, CBO in May (SCD2 point-in-time works)


---
# PART B — SQL

## Q1: 7-day click-to-conversion ROAS per ad

### WHAT TO SAY (the 4 phrases from the prompt):
> "I'm using **LEFT JOIN** to keep the click denominator. I'm guarding with **NULLIF** to avoid div-by-zero. I'm **bounding the joins on time** in the ON clause, not WHERE, to avoid fanout explosion at billions scale. I'm using **COUNT(DISTINCT ...)** instead of COUNT(*) to defend against multi-conversion-per-thread fanout."

### Expected output (from the prompt):
- `ad_10`: 2 clicks → 2 attributed → 2 converted → conv_rate 1.0, revenue 150
- `ad_11`: 1 click → 0 attributed → 0 converted → 0.0, 0

**Note**: the prompt's expected output doesn't include `c5` (April) and doesn't dedup `c99`. In the L6 answer, we **dedup and include the 7-day window** — `c5` is outside the window, `c99` is a duplicate of `c1`.

In [12]:
# Q1: 7-day ROAS — the prompt's SQL, executed
spark.sql("""
    CREATE OR REPLACE TEMP VIEW fct_ad_clicks_dedup AS
    SELECT click_id, ad_id, advertiser_id, click_time, ds
    FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY click_id ORDER BY click_time DESC) rn
          FROM fct_ad_clicks) WHERE rn = 1
""")

q1 = """
    WITH attributed AS (
      SELECT c.ad_id, c.click_id, t.thread_id, t.thread_start_time, c.click_time
      FROM fct_ad_clicks_dedup c
      LEFT JOIN fct_wa_threads t
        ON t.attributed_click_id = c.click_id
       AND t.thread_start_time BETWEEN c.click_time AND c.click_time + INTERVAL 7 DAYS
      WHERE c.ds BETWEEN '2026-05-01' AND '2026-05-08'
    ),
    with_conv AS (
      SELECT a.ad_id, a.click_id, a.thread_id, m.order_value_usd, m.conversion_time
      FROM attributed a
      LEFT JOIN fct_merchant_conversions m
        ON m.thread_id = a.thread_id
       AND m.conversion_time BETWEEN a.click_time AND a.click_time + INTERVAL 7 DAYS
    )
    SELECT ad_id,
           COUNT(DISTINCT click_id) AS clicks,
           COUNT(DISTINCT thread_id) AS attributed_threads,
           COUNT(DISTINCT CASE WHEN order_value_usd > 0 THEN thread_id END) AS converted_threads,
           CAST(COUNT(DISTINCT CASE WHEN order_value_usd > 0 THEN thread_id END) AS DOUBLE)
             / NULLIF(COUNT(DISTINCT click_id), 0) AS conv_rate,
           SUM(order_value_usd) AS total_revenue
    FROM with_conv
    GROUP BY 1
    ORDER BY ad_id
"""
spark.sql(q1).show(truncate=False)

+-----+------+------------------+-----------------+------------------+-------------+
|ad_id|clicks|attributed_threads|converted_threads|conv_rate         |total_revenue|
+-----+------+------------------+-----------------+------------------+-------------+
|ad_10|3     |2                 |2                |0.6666666666666666|150.0        |
|ad_11|1     |0                 |0                |0.0               |NULL         |
|ad_20|1     |1                 |1                |1.0               |200.0        |
+-----+------+------------------+-----------------+------------------+-------------+



## Q2: 5-min reply SLA per advertiser

### WHAT TO SAY:
> "I use **`MIN(CASE WHEN sender_type='user' THEN message_time END)`** to get the first user msg per thread, and same for business. Then `(first_biz - first_user) <= 5min` for the SLA. I **bound by `ds`** in the source table for partition pruning."

**Note**: the prompt's SQL uses `DATE_DIFF('minute', ...)` which is a Hive/Presto function. In PySpark we use `unix_timestamp` for portability. Both are 10/10.

In [13]:
q2 = """
    WITH firsts AS (
      SELECT thread_id,
             MIN(CASE WHEN sender_type='user'     THEN message_time END) AS first_user_time,
             MIN(CASE WHEN sender_type='business' THEN message_time END) AS first_biz_time
      FROM fct_wa_messages_meta
      WHERE ds = '2026-05-01'
      GROUP BY 1
    )
    SELECT t.advertiser_id,
           COUNT(*) AS total_threads,
           AVG((unix_timestamp(first_biz_time) - unix_timestamp(first_user_time))/60.0) AS avg_mins_to_reply,
           SUM(CASE WHEN (unix_timestamp(first_biz_time) - unix_timestamp(first_user_time)) <= 5*60
                    THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS pct_reply_5min
    FROM firsts f
    JOIN fct_wa_threads t ON f.thread_id = t.thread_id
    WHERE first_user_time IS NOT NULL AND first_biz_time IS NOT NULL
    GROUP BY 1
    ORDER BY 1
"""
spark.sql(q2).show(truncate=False)

+-------------+-------------+-----------------+------------------+
|advertiser_id|total_threads|avg_mins_to_reply|pct_reply_5min    |
+-------------+-------------+-----------------+------------------+
|adv_1        |3            |8.3333333333     |0.6666666666666667|
+-------------+-------------+-----------------+------------------+



---
# PART C — CODING (Python)

## Q3: Ghosted threads — the prompt's exact function

### THINK OUT LOUD (the prompt's words):
> "I need to sessionize by `thread_id`. For each thread, find first user message without a business message within 24h window after it. **O(n) single pass if sorted, O(n log n) if not**. Must be streaming-friendly."

### The 3 edge cases the prompt hints at:
1. **Chain**: user sends again before biz replies
2. **Tail**: last user msg never got a reply
3. **Out-of-order** timestamps (always a possibility in distributed logs)

In [14]:
from collections import defaultdict
from datetime import timedelta
from typing import Iterable, Dict, Set, Optional

def find_ghosted_threads(messages: Iterable[Dict],
                         sla: timedelta = timedelta(hours=24),
                         now: Optional[datetime] = None) -> Set[str]:
    """O(n log n) total. Returns set of ghosted thread_ids."""
    msgs = list(messages)
    if not msgs: return set()
    by_thread = defaultdict(list)
    for m in msgs:
        by_thread[m['thread_id']].append(m)
    if now is None:
        now = max(m['message_time'] for m in msgs)
    ghosted = set()
    for tid, thread_msgs in by_thread.items():
        thread_msgs.sort(key=lambda x: x['message_time'])
        last_user_time = None
        has_biz_reply = False
        for m in thread_msgs:
            if m['sender_type'] == 'user':
                if last_user_time and not has_biz_reply \
                   and (m['message_time'] - last_user_time).total_seconds() > 86400:
                    ghosted.add(tid); break
                last_user_time = m['message_time']
                has_biz_reply = False
            else:  # business
                if last_user_time:
                    delta = (m['message_time'] - last_user_time).total_seconds()
                    if delta <= 86400:
                        has_biz_reply = True
        # Tail case: last user msg never got reply
        if last_user_time and not has_biz_reply:
            ghosted.add(tid)
    return ghosted

# ---- 7 tests (the prompt's 2 + 5 more) ----
def test_ghosted():
    # prompt's tests
    msgs1 = [{'thread_id':'t1','sender_type':'user','message_time':datetime(2026,5,1,10,0)},
             {'thread_id':'t1','sender_type':'business','message_time':datetime(2026,5,1,10,4)}]
    assert find_ghosted_threads(msgs1, now=datetime(2026,5,1,11)) == set()  # t1 not ghosted
    msgs2 = [{'thread_id':'t2','sender_type':'user','message_time':datetime(2026,5,1,10,0)}]
    assert find_ghosted_threads(msgs2, now=datetime(2026,5,2,11)) == {'t2'}  # t2 ghosted
    # extra tests
    assert find_ghosted_threads([]) == set()
    msgs3 = [{'thread_id':'t3','sender_type':'user','message_time':datetime(2026,5,1,10)},
             {'thread_id':'t3','sender_type':'user','message_time':datetime(2026,5,1,12)},
             {'thread_id':'t3','sender_type':'business','message_time':datetime(2026,5,1,12,5)}]
    assert find_ghosted_threads(msgs3, now=datetime(2026,5,1,13)) == set()  # chain
    msgs4 = [{'thread_id':'t4','sender_type':'user','message_time':datetime(2026,5,1,10)},
             {'thread_id':'t4','sender_type':'business','message_time':datetime(2026,5,1,10,5)},
             {'thread_id':'t4','sender_type':'user','message_time':datetime(2026,5,2,11)}]
    assert find_ghosted_threads(msgs4, now=datetime(2026,5,2,12)) == {'t4'}  # post-reply
    msgs5 = [{'thread_id':'t5','sender_type':'user','message_time':datetime(2026,5,2,10)},
             {'thread_id':'t5','sender_type':'user','message_time':datetime(2026,5,1,10)},
             {'thread_id':'t5','sender_type':'business','message_time':datetime(2026,5,1,10,5)}]
    assert find_ghosted_threads(msgs5, now=datetime(2026,5,2,11)) == set()  # out-of-order
    print("OK Q3 ghosted threads: 5/5 tests pass")
test_ghosted()

AssertionError: 

## Q4: 10B/day extension (the L6 twist)

### WHAT TO SAY:
> "The Python loop won't scale at 10B/day — driver-side collect is the bottleneck. Use **Spark window `LEAD`** to find the next message in O(n) single pass, no `groupByKey`, no `collect`. Runs in ~5 min on 200 nodes."

In [ ]:
q4 = """
    WITH msgs AS (
      SELECT thread_id, sender_type, message_time,
             LEAD(message_time) OVER (PARTITION BY thread_id ORDER BY message_time) AS next_t,
             LEAD(sender_type)  OVER (PARTITION BY thread_id ORDER BY message_time) AS next_s
      FROM fct_wa_messages_meta
    ),
    flagged AS (
      SELECT thread_id, MAX(
        CASE WHEN next_s IS NULL THEN 1
             WHEN next_s <> 'business' THEN 1
             WHEN (unix_timestamp(next_t) - unix_timestamp(message_time)) > 24*3600 THEN 1
             ELSE 0 END) AS g
      FROM msgs WHERE sender_type='user' GROUP BY thread_id
    )
    SELECT thread_id FROM flagged WHERE g = 1 ORDER BY thread_id
"""
out = spark.sql(q4).collect()
ghosted = [r['thread_id'] for r in out]
print("Q4 ghosted threads via LEAD:", ghosted)
assert ghosted == ['t4'], f"expected ['t4'], got {ghosted}"
print("OK Q4 ghosted threads via LEAD (10B-scale)")

---
# PART D — ARCHITECTURE

## WHAT TO SAY (5 min — sketch this on the whiteboard while talking):

```
FB Ad click -> click_id -> wa.me/?click_id=c1 (ONLY allowed bridge)
                                    |
                                    v
                            WhatsApp Business app
                                    |
                    metadata only  |  business marks ORDER_PLACED
                                    v                              v
                    Scribe -> Kafka (raw_clicks, raw_threads)   merchant_conversion_events Kafka
                                    |                              |
                                    +-----------+------------------+
                                                v
                                    Hourly Spark (Aria):
                                      - dedup ROW_NUMBER on click_id
                                      - stitch attributed_click_id = click_id (NO user_id join)
                                      - join merchant conv with 28-day lookback
                                      - materialize fct_daily_advertiser_roas
                                                |
                                                v
                                       Tectonic (Hive/Parquet)
                                                |
                                +---------------+----------------+
                                v                                v
                        Presto (ad-hoc)                  Laser (real-time)
                        analyst queries                 advertiser dashboard
                                                          15-min refresh
```

## THE 6 L6 TRADE-OFFS (where you win the interview):

1. **Stitching key = `click_id`, NEVER `user_id`**. E2E + GDPR + brand promise. We accept 15-25% organic-untouched threads as `attribution_match_rate`.

2. **Hourly batch, not streaming** for ROAS — 70% cost saving. ROAS doesn't need sub-hour latency. Use **Flink only for real-time ghost alerts**.

3. **28-day rolling re-agg** nightly. Older partitions are immutable. Idempotent `INSERT OVERWRITE PARTITION (ds)`.

4. **Serving: Presto for ad-hoc, Laser for dashboard.** Pre-aggregated counters every 15 min, sub-second reads. No Pinot — write throughput low, read fan-out high.

5. **DQ alerts**: `attribution_rate < 20%` → deep link broken, page ops. `conversion spike > 3σ` → fraud/merchant app bug. `messages row count` dev > 30% → WA API issue.

6. **iOS 14.5 AEA**: Apple limits conversion signals. We use **Aggregated Events API** + WA merchant marks as ground truth. iOS doesn't restrict the merchant's app from logging conversions — that's our most accurate signal.

## CLOSING LINE (memorize — this is the L6 closer):

> "If advertiser complains low match rate, I would add **fallback attribution using coarsened geo-time model with Differential Privacy**, not raw PII join. And I'd propose an **Aggregated Events API** for iOS 14.5 limitations."

In [ ]:
# ---- Q5: materialized fct_daily_advertiser_roas (the serving table) ----
spark.sql("""
    CREATE OR REPLACE TEMP VIEW fct_daily_advertiser_roas AS
    WITH attributed AS (
      SELECT c.advertiser_id, c.ad_id, c.click_id, c.click_time, t.thread_id
      FROM fct_ad_clicks_dedup c
      LEFT JOIN fct_wa_threads t
        ON t.attributed_click_id = c.click_id
       AND t.thread_start_time BETWEEN c.click_time AND c.click_time + INTERVAL 7 DAYS
    )
    SELECT a.advertiser_id, a.ad_id, DATE(a.click_time) AS ds,
           COUNT(DISTINCT a.click_id) AS clicks,
           COUNT(DISTINCT a.thread_id) AS attributed_threads,
           COUNT(DISTINCT CASE WHEN m.order_value_usd > 0 THEN a.thread_id END) AS converted_threads,
           ROUND(SUM(COALESCE(m.order_value_usd, 0)), 2) AS revenue
    FROM attributed a
    LEFT JOIN fct_merchant_conversions m
      ON m.thread_id = a.thread_id
     AND m.conversion_type = 'ORDER_PLACED'
     AND m.conversion_time BETWEEN a.click_time AND a.click_time + INTERVAL 7 DAYS
    GROUP BY a.advertiser_id, a.ad_id, DATE(a.click_time)
""")
spark.sql("SELECT * FROM fct_daily_advertiser_roas ORDER BY ds, advertiser_id, ad_id").show(truncate=False)
by_ad = {r['ad_id']: r.asDict() for r in spark.sql("SELECT * FROM fct_daily_advertiser_roas").collect()}
assert by_ad['ad_10']['revenue'] == 150.0
assert by_ad['ad_11']['revenue'] == 0.0
assert by_ad['ad_20']['revenue'] == 200.0
print("OK Q5 fct_daily_advertiser_roas materialized")

---
# BONUS — Q6: 28-day rolling re-aggregation (late-mark aware)

### WHAT TO SAY:
> "Merchants mark conversions up to 28 days late. So today's ROAS for clicks in the last 28d may change tomorrow. Every night, re-aggregate the last 28 days. Cost: bounded. Older partitions are immutable."

In [ ]:
q6 = """
    WITH params AS (SELECT DATE '2026-05-08' AS run_date),
    clicks_28d AS (
      SELECT c.* FROM fct_ad_clicks_dedup c, params p
      WHERE c.click_time >= p.run_date - INTERVAL 28 DAYS
        AND c.click_time <  p.run_date + INTERVAL 1 DAYS
    ),
    conv_28d AS (
      SELECT m.* FROM fct_merchant_conversions m, params p
      WHERE m.conversion_time >= p.run_date - INTERVAL 28 DAYS
        AND m.conversion_time <  p.run_date + INTERVAL 1 DAYS
        AND m.conversion_type = 'ORDER_PLACED'
    ),
    attributed AS (
      SELECT c.ad_id, c.advertiser_id, c.click_id, c.click_time, t.thread_id
      FROM clicks_28d c
      LEFT JOIN fct_wa_threads t
        ON t.attributed_click_id = c.click_id
       AND t.thread_start_time BETWEEN c.click_time AND c.click_time + INTERVAL 7 DAYS
    )
    SELECT a.ad_id, a.advertiser_id,
           COUNT(DISTINCT a.click_id) AS clicks,
           COUNT(DISTINCT CASE WHEN m.order_value_usd > 0 THEN a.thread_id END) AS converted,
           ROUND(SUM(COALESCE(m.order_value_usd, 0)), 2) AS revenue_28d
    FROM attributed a
    LEFT JOIN conv_28d m
      ON m.thread_id = a.thread_id
     AND m.conversion_time BETWEEN a.click_time AND a.click_time + INTERVAL 7 DAYS
    GROUP BY a.ad_id, a.advertiser_id
    ORDER BY a.ad_id
"""
spark.sql(q6).show(truncate=False)

---
# BONUS — Q7: Idempotent dedup with ROW_NUMBER

### WHAT TO SAY:
> "All 3 fact tables have idempotency at ingest. Dedup with `ROW_NUMBER() OVER (PARTITION BY unique_key ORDER BY processing_time DESC) = 1`. In streaming, dedup in Flink state store keyed on the same unique key."

In [ ]:
from pyspark.sql.functions import current_timestamp, row_number
from pyspark.sql.window import Window

fct_ad_clicks.withColumn("processing_time", current_timestamp()).createOrReplaceTempView("fct_ad_clicks_with_proc")
deduped_count = spark.sql("""
    SELECT COUNT(*) AS n FROM (
      SELECT *, ROW_NUMBER() OVER (PARTITION BY click_id ORDER BY processing_time DESC) rn
      FROM fct_ad_clicks_with_proc
    ) WHERE rn = 1
""").collect()[0]['n']
raw_count = fct_ad_clicks.count()
print(f"Raw: {raw_count}, Deduped: {deduped_count} (c99 deduped to 1)")
assert deduped_count == 5  # c1, c2, c3, c4, c5, c99 -> 5 distinct
print("OK Q7 idempotent dedup")

---
# LIVE DRILL — Try it yourself

**Interviewer gives you this table on the board. You have 5 minutes. Write the SQL, then I'll poke holes like the interviewer will.**

**Q**: For each advertiser, find the % of *organic* threads (attributed_click_id IS NULL) where business replied within 5 min. Return: advertiser_id, total_organic_threads, pct_5min.

In [ ]:
# ---- YOUR ANSWER HERE (try before looking at the model answer below) ----
your_sql = """
-- Write your SQL here
SELECT 1
"""
# spark.sql(your_sql).show(truncate=False)  # uncomment to test

In [ ]:
# ---- MODEL ANSWER (10/10) ----
model_sql = """
    WITH organic AS (
      SELECT thread_id, advertiser_id
      FROM fct_wa_threads
      WHERE attributed_click_id IS NULL    -- organic only
    ),
    firsts AS (
      SELECT thread_id,
             MIN(CASE WHEN sender_type='user'     THEN message_time END) AS first_user_time,
             MIN(CASE WHEN sender_type='business' THEN message_time END) AS first_biz_time
      FROM fct_wa_messages_meta
      WHERE ds BETWEEN '2026-05-01' AND '2026-05-31'
      GROUP BY thread_id
    )
    SELECT o.advertiser_id,
           COUNT(*) AS total_organic_threads,
           ROUND(SUM(CASE WHEN f.first_user_time IS NOT NULL
                            AND f.first_biz_time IS NOT NULL
                            AND (unix_timestamp(f.first_biz_time) - unix_timestamp(f.first_user_time)) <= 5*60
                           THEN 1 ELSE 0 END) * 1.0 / COUNT(*), 3) AS pct_5min
    FROM organic o
    LEFT JOIN firsts f USING (thread_id)
    GROUP BY o.advertiser_id
    ORDER BY o.advertiser_id
"""
spark.sql(model_sql).show(truncate=False)

## Interviewer pokes holes (this is what L6 does):

1. **"What if a thread has no messages? It still counts in total_organic_threads, which drags the % down. Is that what you want?"**
> Counter-argument: yes, ghost/no-message organic threads SHOULD drag the % — it's the product signal. But if the metric is for a dashboard, filter to threads-with-messages: `WHERE first_user_time IS NOT NULL`.

2. **"What about threads where business messages first (user reads it but never replies)?"**
> Not a ghost, not a 5-min reply. Goes to a separate metric: 'proactive business reach'.

3. **"Why LEFT JOIN from organic?"**
> To keep threads-with-no-messages in the denominator. If we INNER JOIN, we silently drop them and inflate the %.

4. **"What if business messages multiple times in 5 min?"**
> `MIN(CASE WHEN sender_type='business'...)` takes the first one. That's correct — we want time-to-FIRST-reply.

5. **"What if `message_time` has clock skew across regions?"**
> In production, normalize to UTC at ingest. Scribe uses NTP-synced clocks; if skew > 1s, page on it. For a 5-min SLA, 1s skew is noise.

---
# CHEAT SHEET — 10/10 L6 MEMORIZATION

## The 5-sentence opener (memorize word-for-word, 30 sec to say):
1. "Success is **advertiser ROI**: conversion rate, time-to-first-reply, order value."
2. "Constraint: **E2E encrypted** — only metadata, no content."
3. "Stitching via **`click_id`**, never user_id."
4. "**Late data is the norm** — 28-day lookback re-aggregation."
5. "**Idempotency** at ingest via `ROW_NUMBER` dedup."

## The 5 hard-coded SQL snippets (memorize the shape, not syntax):
| Concept | Recite |
|---|---|
| **No div-by-zero** | `CAST(num AS DOUBLE) / NULLIF(denom, 0)` |
| **Bounded join** | `LEFT JOIN x ON key AND t BETWEEN t0 AND t0 + INTERVAL 7 DAYS` |
| **Dedup** | `ROW_NUMBER() OVER (PARTITION BY click_id ORDER BY processing_time DESC) = 1` |
| **First per role** | `MIN(CASE WHEN sender_type='user' THEN ts END) OVER (PARTITION BY thread_id)` |
| **Anti-join (NULL-safe)** | `WHERE NOT EXISTS (...)` — never `NOT IN` if subquery can have NULL |

## The 4 phrases from the prompt (the L6 what-to-say-while-coding):
- *"I'm using **LEFT JOIN** to keep the click denominator."*
- *"I'm guarding with **NULLIF** to avoid div-by-zero."*
- *"I'm **bounding the joins on time** in the ON clause to avoid fanout explosion at billions scale."*
- *"I'm using **COUNT(DISTINCT ...)** to defend against multi-conversion-per-thread fanout."*

## The 7 L6 phrases to say out loud:
- *"Denominator is clicks, not threads."*
- *"Bounded the join on time to prevent fanout at billions scale."*
- *"28-day lookback handles late merchant marks."*
- *"E2E means no PII join — only `click_id`."*
- *"Partition by `ds`, cluster by `advertiser_id`."*
- *"SCD2 on `dim_ad` for budget_type history."*
- *"Hourly Spark, not Flink — 70% cost saving."*

## The closing line (memorize):
> "If advertiser complains low match rate, I would add fallback attribution using coarsened geo-time model with Differential Privacy, not raw PII join. And I'd propose an Aggregated Events API for iOS 14.5 limitations."

## The practice tip (do this on the whiteboard):
> When they show you a table, **immediately redraw it as Input -> Output on your side**. It buys you 30 seconds of think-time and shows structured thinking.

## 2-week drill plan:
- **Day 1-3**: Run each cell, change 1 number, re-run, predict result.
- **Day 4-7**: Cover the SQL in Q1-Q7, rewrite from memory.
- **Day 8-10**: Cover the Python in Q3, time yourself (12 min including the 5 tests).
- **Day 11-14**: Practice the 5-sentence opener + 7 L6 phrases + closing line out loud. Record. Listen back.

In [ ]:
spark.stop()
print("\n=== ALL TESTS PASSED ===")